In [36]:
%pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [37]:
%pip install streamlit
import os
import streamlit as st
from dotenv import load_dotenv

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

Note: you may need to restart the kernel to use updated packages.


In [38]:
from pypdf import PdfReader

def read_pdfs(pdf_files):
    all_text = ""

    for pdf in pdf_files:
        reader = PdfReader(pdf)

        for page in reader.pages:
            text = page.extract_text()

            if text:
                all_text += text

    return all_text

In [39]:
text=read_pdfs(['attention.pdf'])

In [40]:
print(text)

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exper

In [41]:
def split_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    return splitter.split_text(text)

In [42]:
chunks = split_text(text)

In [43]:
def create_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )


In [44]:
def build_vector_store(chunks):
    embeddings = create_embeddings()

    documents = [
        Document(page_content=chunk) for chunk in chunks
    ]

    vector_store = FAISS.from_documents(
        documents, embeddings
    )

    vector_store.save_local("faiss_index")

In [45]:
from pypdf import PdfReader

reader = PdfReader("attention.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text() or ""

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(text)

print(len(chunks))

50


In [47]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(text)

print(len(chunks))

50


In [48]:
def load_vector_store():
    embeddings = create_embeddings()
    return FAISS.load_local(
        "faiss_index",
        embeddings,
        allow_dangerous_deserialization=True
    )

In [49]:
def retrieve_chunks(questions):
    vector_store =load_vector_store()
    return vector_store.similarity_search(question,k=4)

In [50]:
def retrieve_chunks(question):
    vector_store = load_vector_store()
    return vector_store.similarity_search(question, k=4)

In [51]:
docs = retrieve_chunks("what is self attention?")

for d in docs:
    print(d.page_content[:1000])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 720.48it/s]


is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same place, in this case, the output of the previous layer in the
encoder. Each position in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
relative positions, since for any fixed offsetk,PE pos+k can be

In [53]:

def get_prompt_and_llm():
    prompt_template = """
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present in the context, say exactly:
"The answer is not available in the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["context", "question"]
    )

    llm = ChatGoogleGenerativeAI(
        model="models/gemini-flash-lite-latest",
        temperature=1
    )

    return prompt, llm

In [54]:

def answer_question(question):
    docs = retrieve_chunks(question)
    
    context = "\n\n".join([doc.page_content for doc in docs])
    
    prompt, llm = get_prompt_and_llm()
    final_prompt = prompt.format(context=context, question=question)
    
    response = llm.invoke(final_prompt)
    return response.content

In [55]:
answer_question("what is self attention?")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1249.16it/s]


[{'type': 'text',
  'text': 'Based on the provided context, self-attention (sometimes called intra-attention) is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. In a self-attention layer, all of the keys, values, and queries come from the same place.',
  'extras': {'signature': 'EmAKXgFpFH0T4toyBJ2meXJPc26ut9SafYLLmBTLLPSB4T8iwBhmTxuqr40KCvDfM1z0t7OqONAE9rcd8ialAZAGczFTjSGj9bfg9Ha8Txwh5N43lKmtK1U5GzkwHDlL4q8='}}]